# 🏆 IFest DAC 2026 — Data Analysis Competition (Penyisihan)
## Deteksi Inkongruensi Berita Bahasa Indonesia (Macro F1-Score)

**Pipeline Opsi A: Murni Algoritmik & Patuh Regulasi (Discourse-Aware NLP, Adaptive Inversion Scopes, & Multi-Model Stacking Ensemble)**

Notebook ini mengimplementasikan solusi klasifikasi biner untuk mendeteksi ketidaksesuaian (*incongruence / clickbait*) antara judul dan isi berita berbahasa Indonesia. Solusi ini dirancang secara murni berbasis rekayasa 45 fitur diskursus-leksikal adaptif dan pembelajaran mesin ensemble klasik yang mematuhi secara ketat seluruh regulasi resmi kompetisi:
1. **Rule #4**: *No Pretrained Models* (bebas IndoBERT, Word2Vec/FastText pretrained, dll.).
2. **Rule #5**: *No Generative AI / LLM APIs* (bebas OpenAI, Gemini, Claude, dll.).
3. **Rule #8**: *No Data Leakage* (fitur statistik & LSA di-fit secara ketat HANYA pada data latih; evaluasi per-sampel murni adaptif terhadap data baru).
4. **Rule #11**: *Deterministic & Reproducible* (`SEED = 42`).

In [1]:
import os
import random
import re
import math
from collections import Counter
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier, ExtraTreesClassifier
from sklearn.metrics import f1_score, classification_report, confusion_matrix
import xgboost as xgb

# Menetapkan Random Seed untuk Reproducibility (Kewajiban Guidebook Hal. 5 Poin 6 & Rule #11)
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

print("All libraries imported successfully! Deterministic Random Seed set to 42.")

All libraries imported successfully! Deterministic Random Seed set to 42.

## 1. Load Data & Eksplorasi Distribusi
Dataset terdiri dari 14.397 data latih (`train.csv`) dan 3.603 data uji (`test.csv`).

In [2]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

print(f"Train shape: {train.shape}")
print(f"Test shape:  {test.shape}")

print("\n--- Distribusi Label Train ---")
print(train['label'].value_counts())
print(f"Rasio Class 1: {(train['label']==1).mean()*100:.2f}%")
print(f"Rasio Class 0: {(train['label']==0).mean()*100:.2f}%")

Train shape: (14397, 4)
Test shape:  (3603, 3)

--- Distribusi Label Train ---
label
1    12960
0     1437
Name: count, dtype: int64
Rasio Class 1: 90.02%
Rasio Class 0: 9.98%

## 2. Kamus Leksikal, Entitas Kanonikal, & Skop Negasi Adaptif
Modul ini mendefinisikan kamus leksikal terkurasi untuk menangani variasi penulisan bahasa Indonesia:
- **Kamus Lokasi & Normalisasi**: Menangani singkatan wilayah (*jabar*, *jakpus*, *diy*, dll.) ke bentuk kanonikal.
- **Kamus Tokoh Publik & Kanonikalisasi Alias**: Menghubungkan alias (*jokowi*, *prabowo*, *terawan*, dll.) ke nama standar.
- **Daftar Lembaga & Penanda Entitas Komposit**: Mengenali kementerian, institusi riset/kampus (*ugm*, *ui*, *kemenkes*, dll.) dan penanda institusi wilayah (*bea cukai*, *rsud*, *polres*, dll.).
- **Indikator Negasi, Verba Inheren, & Antonim**: Menangkap pembalikan makna faktual (*tak*, *belum*, *tolak*, dll.), verba negatif (*melarang*, *membatalkan*, *menggagalkan*, dll.), dan pasangan antonim polaritas.
- **Detektor Sanggahan & Unit Angka**: Mengidentifikasi leksikon sanggahan berita (*Riedel et al.*) dan pasangan angka beserta satuannya.

In [3]:
import re

# 1. Indonesian Stopwords (Kompak & Informatif)
STOP_WORDS = {
    'yang', 'di', 'dan', 'ini', 'dari', 'dalam', 'untuk', 'pada', 'ke', 'karena',
    'oleh', 'saat', 'ada', 'akan', 'juga', 'dengan', 'adalah', 'ia', 'itu', 'atau',
    'bisa', 'tersebut', 'bukan', 'tidak', 'tak', 'namun', 'mereka', 'kita', 'para',
    'lagi', 'saja', 'sudah', 'secara', 'harus', 'hanya', 'agar', 'bagi', 'bahwa',
    'terus', 'setelah', 'terkait', 'tentang', 'atas', 'olehnya', 'menjadi', 'sebagai',
    'masih', 'bila', 'belum', 'banyak', 'sangat', 'tahun', 'hari', 'bulan', 'lalu',
    'seperti', 'antara', 'dapat', 'mengatakan', 'ujar', 'kata', 'ungkap', 'tambah',
    'jelas', 'pungkas', 'tutur', 'sebut', 'beber', 'imbuh'
}

# 2. Pasangan Antonim & Konflik Semantik
EXPANDED_ANTONYM_PAIRS = [
    ('positif', 'negatif'), ('naik', 'turun'), ('lonjakan', 'penurunan'),
    ('buka', 'tutup'), ('dibuka', 'ditutup'), ('tambah', 'kurang'),
    ('tinggi', 'rendah'), ('meningkat', 'menurun'), ('sembuh', 'meninggal'),
    ('hidup', 'mati'), ('masuk', 'keluar'), ('zona merah', 'zona hijau'),
    ('sukses', 'gagal'), ('untung', 'rugi'), ('kalah', 'menang'),
    ('benar', 'salah'), ('palsu', 'asli'), ('hoaks', 'fakta')
]

# 3. Leksikon Sanggahan (Refuting Words - Riedel et al. FNC-1)
REFUTING_PHRASES = [
    'tidak benar', 'tak benar', 'bukan', 'keliru', 'hoaks', 'hoax',
    'bohong', 'bantah', 'membantah', 'dibantah', 'fiktif', 'palsu',
    'disinformasi', 'misinformasi', 'manipulasi', 'klaim sepihak'
]
re_refuting = re.compile(r'\b(?:' + '|'.join(re.escape(w) for w in REFUTING_PHRASES) + r')\b', re.I)
re_num_unit = re.compile(r'(\b\d+[\d\.,]*)\s+([a-zA-Z]+)')

# 4. Kamus Wilayah / Lokasi & Kanonikalisasi Alias
CANONICAL_LOC = {
    'jabar': 'jawa barat', 'jateng': 'jawa tengah', 'jatim': 'jawa timur',
    'diy': 'yogyakarta', 'jogja': 'yogyakarta', 'jogjakarta': 'yogyakarta',
    'dki': 'dki jakarta', 'jakpus': 'jakarta pusat', 'jakbar': 'jakarta barat',
    'jaksel': 'jakarta selatan', 'jaktim': 'jakarta timur', 'jakut': 'jakarta utara',
    'surakarta': 'solo', 'ina': 'indonesia', 'ri': 'indonesia'
}

ALL_LOCATIONS = [
    'deli serdang', 'tebing tinggi', 'pematang siantar', 'pematangsiantar', 'tanjung balai', 'tanjungbalai',
    'bandar lampung', 'padang panjang', 'pesisir selatan', 'tanah datar', 'pagar alam', 'ogan ilir',
    'muara enim', 'banyuasin', 'kutai kartanegara', 'sulawesi tenggara', 'sulawesi selatan', 'sulawesi utara',
    'sulawesi tengah', 'sulawesi barat', 'sumatera utara', 'sumatera barat', 'sumatera selatan',
    'jawa timur', 'jawa barat', 'jawa tengah', 'nusa tenggara barat', 'nusa tenggara timur',
    'kalimantan timur', 'kalimantan barat', 'kalimantan selatan', 'kalimantan tengah', 'kalimantan utara',
    'dki jakarta', 'luwu utara', 'luwu timur', 'konawe utara', 'konawe selatan', 'musi banyuasin',
    'toba samosir', 'gunung kidul', 'gunungkidul', 'kulon progo', 'kulonprogo',
    'jakarta pusat', 'jakarta barat', 'jakarta selatan', 'jakarta timur', 'jakarta utara',
    'jakpus', 'jakbar', 'jaksel', 'jaktim', 'jakut',
    'jakarta', 'jabar', 'jateng', 'jatim', 'banten', 'jogja', 'yogyakarta', 'diy',
    'bandung', 'surabaya', 'semarang', 'bogor', 'depok', 'tangerang', 'bekasi',
    'solo', 'surakarta', 'malang', 'serang', 'cilegon', 'pandeglang', 'lebak',
    'sukabumi', 'cianjur', 'purwakarta', 'karawang', 'subang', 'indramayu',
    'cirebon', 'majalengka', 'sumedang', 'kuningan', 'ciamis', 'tasikmalaya',
    'garut', 'banjar', 'pangandaran', 'cimahi', 'brebes', 'tegal', 'slawi',
    'pemalang', 'pekalongan', 'batang', 'kendal', 'demak', 'kudus', 'jepara',
    'pati', 'rembang', 'blora', 'grobogan', 'purwodadi', 'sragen', 'karanganyar',
    'wonogiri', 'sukoharjo', 'klaten', 'boyolali', 'magelang', 'muntilan',
    'temanggung', 'wonosobo', 'purworejo', 'kebumen', 'banyumas', 'purwokerto',
    'purbalingga', 'banjarnegara', 'cilacap', 'salatiga', 'sleman', 'bantul',
    'wates', 'wonosari', 'pacitan', 'ponorogo',
    'trenggalek', 'tulungagung', 'blitar', 'kediri', 'nganjuk', 'madiun',
    'magetan', 'ngawi', 'bojonegoro', 'tuban', 'lamongan', 'gresik', 'bangkalan',
    'sampang', 'pamekasan', 'sumenep', 'sidoarjo', 'mojokerto', 'jombang',
    'pasuruan', 'probolinggo', 'lumajang', 'bondowoso', 'situbondo', 'jember',
    'banyuwangi', 'batu',
    'aceh', 'sumut', 'sumbar', 'riau', 'kepri', 'jambi', 'bengkulu', 'sumsel',
    'babel', 'lampung', 'medan', 'padang', 'palembang', 'pekanbaru', 'batam',
    'tanjungpinang', 'metro', 'pangkalpinang', 'binjai', 'asahan',
    'karo', 'simalungun', 'tapanuli', 'toba', 'mandailing', 'langkat', 'nias',
    'labuhanbatu', 'lhokseumawe', 'langsa', 'sabang', 'pidie', 'meulaboh',
    'bukittinggi', 'payakumbuh', 'pariaman', 'solok', 'sawahlunto', 'agam',
    'pasaman', 'dumai', 'bengkalis', 'siak', 'kampar', 'rokan', 'indragiri',
    'pelalawan', 'bintan', 'karimun', 'natuna', 'anambas', 'lingga', 'prabumulih',
    'lubuklinggau', 'lahat', 'musi', 'bangka', 'belitung',
    'kalbar', 'kalteng', 'kalsel', 'kaltim', 'kaltara', 'pontianak', 'singkawang',
    'palangkaraya', 'banjarmasin', 'banjarbaru', 'samarinda', 'balikpapan',
    'bontang', 'tarakan', 'sambas', 'sintang', 'ketapang', 'kapuas', 'kotawaringin',
    'barito', 'kutai', 'berau', 'pasir', 'penajam', 'nunukan', 'bulungan', 'malinau',
    'sulsel', 'sulut', 'sulteng', 'sultra', 'sulbar', 'gorontalo', 'makassar',
    'manado', 'palu', 'kendari', 'mamuju', 'parepare', 'palopo',
    'bitung', 'tomohon', 'kotamobagu', 'bau-bau', 'gowa', 'maros', 'bone',
    'luwu', 'wajo', 'soppeng', 'bulukumba', 'bantaeng', 'selayar', 'toraja',
    'enrekang', 'sidrap', 'pinrang', 'minahasa', 'sangihe', 'talaud', 'poso',
    'donggala', 'banggai', 'morowali', 'tolitoli', 'buol', 'buton', 'muna',
    'kolaka', 'konawe', 'wakatobi',
    'bali', 'denpasar', 'badung', 'gianyar', 'tabanan', 'buleleng', 'jembrana',
    'karangasem', 'klungkung', 'bangli', 'mataram', 'bima', 'lombok', 'sumbawa',
    'dompu', 'kupang', 'timor', 'belu', 'flores', 'ende', 'manggarai', 'sumba',
    'alor', 'rote', 'ambon', 'tual', 'ternate', 'tidore', 'seram', 'buru', 'halmahera',
    'jayapura', 'sorong', 'manokwari', 'merauke', 'mimika', 'timika', 'biak',
    'nabire', 'fakfak', 'wamena', 'papua',
    'amerika', 'inggris', 'jerman', 'prancis', 'china', 'cina', 'jepang', 'korsel',
    'brasil', 'brazil', 'meksiko', 'italia', 'spanyol', 'rusia', 'afsel', 'afrika',
    'australia', 'singapura', 'malaysia', 'filipina', 'thailand', 'vietnam', 'india',
    'arab', 'saudi', 'turki', 'iran', 'irak', 'israel', 'palestina', 'mesir',
    'belanda', 'portugal', 'kanada', 'selandia', 'ukraina', 'lebanon', 'indonesia', 'ri'
]
LOC_REGEX = re.compile(r'\b(' + '|'.join(re.escape(loc) for loc in sorted(ALL_LOCATIONS, key=len, reverse=True)) + r')\b', re.I)

# 5. Kamus Tokoh Publik & Kanonikalisasi Alias
CANONICAL_FIGURES = {
    'joko widodo': 'jokowi', 'maruf amin': 'ma ruf', 'anies baswedan': 'anies',
    'prabowo subianto': 'prabowo', 'sandiaga uno': 'sandiaga', 'ridwan kamil': 'rk',
    'ganjar pranowo': 'ganjar', 'airlangga hartarto': 'airlangga', 'luhut pandjaitan': 'luhut',
    'mahfud md': 'mahfud', 'muhadjir effendy': 'muhadjir', 'ahmad riza patria': 'riza patria',
    'gibran rakabuming': 'gibran', 'bima arya sugiarto': 'bima arya', 'wiku adisasmito': 'wiku',
    'achmad yurianto': 'yurianto', 'agus harimurti': 'ahy', 'agus harimurti yudhoyono': 'ahy',
    'susilo bambang yudhoyono': 'sby', 'puan maharani': 'puan', 'donald trump': 'trump',
    'joe biden': 'biden', 'vladimir putin': 'putin'
}
PUBLIC_FIGURES = list(set(list(CANONICAL_FIGURES.keys()) + list(CANONICAL_FIGURES.values()) + [
    'terawan', 'budi gunadi', 'erick thohir', 'sri mulyani', 'dedie rachim',
    'bobby nasution', 'doni monardo', 'idham azis', 'listyo sigit', 'hadi tjahjanto',
    'nadiem', 'nadiem makarim', 'megawati', 'rizieq', 'rizieq shihab',
    'sinovac', 'pfizer', 'moderna', 'astrazeneca', 'novavax', 'bio farma'
]))
FIGURE_REGEX = re.compile(r'\b(' + '|'.join(re.escape(f) for f in sorted(PUBLIC_FIGURES, key=len, reverse=True)) + r')\b', re.I)

# 6. Lembaga, Institusi, & Penanda Entitas Komposit
INSTITUTIONS = {
    'itb', 'ugm', 'ui', 'unpad', 'undip', 'ipb', 'unair', 'its', 'unhas', 'unsri', 'usu',
    'uny', 'stie', 'ekuitas', 'kemenkes', 'menristek', 'bnpb', 'satgas', 'kpu', 'gerindra',
    'dpc', 'dprd', 'dpr', 'polri', 'bareskrim', 'dpmptsp', 'menko', 'pmk', 'polda', 'polres',
    'kpk', 'bpom', 'who', 'mui', 'kemenhub', 'kemendagri', 'kemensos', 'kemendikbud'
}

INSTITUTION_MARKERS = [
    'bea cukai', 'kantor bea cukai', 'rsud', 'puskesmas', 'dinkes', 'dinas kesehatan',
    'pemprov', 'pemkot', 'pemkab', 'polres', 'polda', 'kodim', 'pasar',
    'bupati', 'walikota', 'gubernur', 'kapolres', 'dprd', 'sekolah'
]
re_inst_marker = re.compile(r'\b(' + '|'.join(re.escape(m) for m in sorted(INSTITUTION_MARKERS, key=len, reverse=True)) + r')\b', re.I)

# 7. Kata Negasi & Verba Negatif Inheren
NEGATION_WORDS = {
    'tak', 'tidak', 'bukan', 'belum', 'tanpa', 'gagal', 'batal', 'larang', 'tolak',
    'bebas', 'nihil', 'nol', 'stop', 'hentikan', 'anti', 'menolak', 'membantah',
    'menyanggah', 'menafikan', 'mustahil'
}
NEG_WORDS = {'tak', 'tidak', 'bukan', 'belum', 'tanpa', 'gagal', 'batal', 'larang', 'tolak', 'anti', 'mustahil'}

INHERENT_NEG_VERBS = {
    'larang', 'melarang', 'dilarang', 'batal', 'membatalkan', 'dibatalkan',
    'gagal', 'menggagalkan', 'digagalkan', 'tolak', 'menolak', 'ditolak',
    'bantah', 'membantah', 'dibantah', 'tunda', 'menunda', 'ditunda'
}

PREFIXES = ['meng', 'meny', 'men', 'mem', 'me', 'di', 'ber', 'ter', 'pen', 'pem', 'peny', 'pe', 'ke', 'se']
SUFFIXES = ['kan', 'an', 'i', 'nya', 'lah', 'kah']

def light_stem(word):
    if len(word) <= 4:
        return word
    w = word
    for pref in PREFIXES:
        if w.startswith(pref) and len(w) - len(pref) >= 3:
            w = w[len(pref):]
            break
    for suff in SUFFIXES:
        if w.endswith(suff) and len(w) - len(suff) >= 3:
            w = w[:-len(suff)]
            break
    return w

def get_canonical_locations(text):
    raw_locs = [m.group(0).lower() for m in LOC_REGEX.finditer(text)]
    return set(CANONICAL_LOC.get(loc, loc) for loc in raw_locs)

def get_canonical_figures(text):
    raw_figs = [m.group(0).lower() for m in FIGURE_REGEX.finditer(text)]
    return set(CANONICAL_FIGURES.get(f, f) for f in raw_figs)

def extract_numbers_comprehensive(text):
    norm = re.sub(r'(\d+)\s+(\d+)', r'\1\2', text)
    num_matches = set(re.findall(r'\b\d+(?:[.,]\d+)?%?\b', norm))
    return {n for n in num_matches if not (len(n) == 4 and n.isdigit() and n.startswith(('19', '20')))}

def clean_title(text):
    return re.sub(r'\s+', ' ', str(text)).strip()

print("Modul leksikal kanonikal, ekspansi antonim, leksikon sanggahan, parser angka, dan skop negasi/entitas komposit siap!")

Modul leksikal kanonikal, ekspansi antonim, leksikon sanggahan, dan parser angka siap!

## 3. Ekstraksi Fitur Leksikal & Okapi BM25 (*Strict Train-Only Fitting*)
Sesuai **Rule #8 (No Data Leakage)**, seluruh vectorizer TF-IDF dan Okapi BM25 **hanya di-fit pada data train** (`train.csv`), kemudian ditransformasikan ke data train dan test.

In [4]:
from sklearn.decomposition import TruncatedSVD

if 'clean_title' not in globals():
    def clean_title(text):
        return re.sub(r'\s+', ' ', str(text)).strip()

train_titles = [clean_title(t) for t in train['title'].fillna('').astype(str).tolist()]
train_contents = train['content'].fillna('').astype(str).tolist()
train_fs = [" ".join(c.split()[:40]) for c in train_contents]

test_titles = [clean_title(t) for t in test['title'].fillna('').astype(str).tolist()]
test_contents = test['content'].fillna('').astype(str).tolist()
test_fs = [" ".join(c.split()[:40]) for c in test_contents]

# TF-IDF Vectorization
print("Fitting TF-IDF Word & Char Vectorizers (Strict Train-Only Fitting)...")
tfidf_word = TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_features=15000, sublinear_tf=True)
tfidf_word.fit(train_titles + train_contents)

tfidf_char = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), min_df=5, max_features=15000, sublinear_tf=True)
tfidf_char.fit(train_titles + train_contents)

# Latent Semantic Analysis (TruncatedSVD 100 Components - Rule #4 Compliant)
print("Fitting TruncatedSVD LSA (100 components)...")
svd = TruncatedSVD(n_components=100, random_state=SEED)
train_c_w = tfidf_word.transform(train_contents)
svd.fit(train_c_w)

def compute_corpus_stats(tfidf_vec, corpus_texts):
    X = tfidf_vec.transform(corpus_texts)
    doc_lens = np.asarray(X.sum(axis=1)).flatten()
    avg_dl = float(np.mean(doc_lens))
    idfs = tfidf_vec.idf_
    return avg_dl, idfs

print("Computing BM25 & Pairwise Similarity Matrices...")
avg_dl_c, idf_w = compute_corpus_stats(tfidf_word, train_contents)
avg_dl_fs, _ = compute_corpus_stats(tfidf_word, train_fs)

def calc_bm25_score(t_vec, c_vec, c_len, avg_dl, k1=1.5, b=0.75):
    c_indices = c_vec.indices
    c_data = c_vec.data
    t_indices = t_vec.indices
    t_data = t_vec.data
    if len(t_indices) == 0 or len(c_indices) == 0:
        return 0.0
    c_dict = dict(zip(c_indices, c_data))
    t_dict = dict(zip(t_indices, t_data))
    common = set(t_indices).intersection(c_indices)
    if not common:
        return 0.0
    score = 0.0
    len_norm = 1.0 - b + b * (c_len / (avg_dl + 1e-9))
    for idx in common:
        tf_c = c_dict[idx]
        tf_t = t_dict[idx]
        idf_val = idf_w[idx] if idx < len(idf_w) else 1.0
        term_score = idf_val * (tf_c * (k1 + 1.0)) / (tf_c + k1 * len_norm) * tf_t
        score += term_score
    return float(score)

def get_pairwise_vectors(titles, contents, fs_texts):
    t_w = tfidf_word.transform(titles)
    c_w = tfidf_word.transform(contents)
    fs_w = tfidf_word.transform(fs_texts)
    
    t_c = tfidf_char.transform(titles)
    c_c = tfidf_char.transform(contents)
    fs_c = tfidf_char.transform(fs_texts)
    
    cos_w_c = np.asarray((t_w.multiply(c_w)).sum(axis=1)).flatten()
    cos_w_fs = np.asarray((t_w.multiply(fs_w)).sum(axis=1)).flatten()
    cos_c_c = np.asarray((t_c.multiply(c_c)).sum(axis=1)).flatten()
    cos_c_fs = np.asarray((t_c.multiply(fs_c)).sum(axis=1)).flatten()
    
    t_lsa = svd.transform(t_w)
    c_lsa = svd.transform(c_w)
    fs_lsa = svd.transform(fs_w)
    
    cos_lsa_c = (t_lsa * c_lsa).sum(axis=1) / (np.linalg.norm(t_lsa, axis=1) * np.linalg.norm(c_lsa, axis=1) + 1e-9)
    cos_lsa_fs = (t_lsa * fs_lsa).sum(axis=1) / (np.linalg.norm(t_lsa, axis=1) * np.linalg.norm(fs_lsa, axis=1) + 1e-9)
    
    c_lens = np.asarray(c_w.sum(axis=1)).flatten()
    fs_lens = np.asarray(fs_w.sum(axis=1)).flatten()
    bm25_c = [calc_bm25_score(t_w[i], c_w[i], c_lens[i], avg_dl_c) for i in range(len(titles))]
    bm25_fs = [calc_bm25_score(t_w[i], fs_w[i], fs_lens[i], avg_dl_fs) for i in range(len(titles))]
    
    return cos_w_c, cos_w_fs, cos_c_c, cos_c_fs, np.array(bm25_c), np.array(bm25_fs), cos_lsa_c, cos_lsa_fs, t_w

cos_w_c_tr, cos_w_fs_tr, cos_c_c_tr, cos_c_fs_tr, bm25_c_tr, bm25_fs_tr, cos_lsa_c_tr, cos_lsa_fs_tr, tr_t_w = get_pairwise_vectors(train_titles, train_contents, train_fs)
cos_w_c_te, cos_w_fs_te, cos_c_c_te, cos_c_fs_te, bm25_c_te, bm25_fs_te, cos_lsa_c_te, cos_lsa_fs_te, te_t_w = get_pairwise_vectors(test_titles, test_contents, test_fs)

print("TF-IDF, Okapi BM25, dan TruncatedSVD LSA (100 komponen) berhasil di-fit secara ketat HANYA pada data train!")

TF-IDF, Okapi BM25, dan TruncatedSVD LSA berhasil di-fit secara ketat HANYA pada data train!

## 4. Pipeline Rekayasa 45 Fitur Optimal (Opsi A: Multi-Scale Windows, Fact Guards, & Adaptive Inversion Scopes)
Fitur diekstrak secara murni per pasangan artikel tanpa ketergantungan antar-baris data uji:
1. **Global & Lead Similarities**: TF-IDF Word, Char, SVD/LSA Cosine (100 dimensi), dan Okapi BM25 pada konten penuh & lead window.
2. **Multi-Scale Sliding Windows**: Fixed token windows (25, 50, 80 token) menangkap segmen konten paling relevan tanpa bias panjang teks.
3. **Discourse Position & Focal Window**: Posisi jendela terbaik (`best_win_pos_norm`), gap lead vs best (`best_minus_lead`), kontras top-1 vs top-2 window, dan kemiripan TF-IDF jendela fokal terbaik.
4. **Targeted Fact Guards**:
   - `head_entity_absence`: Mendeteksi ketiadaan entitas/tokoh utama di awal judul dari seluruh isi berita.
   - `num_unit_match_ratio`: Memverifikasi kehadiran pasangan angka dan satuan di dalam konten.
   - `lead_negation_asymmetry`: Menangkap pembalikan fakta arah (*title negates vs lead affirms*).
5. **Literature-Backed Features**:
   - `refuting_score`: Deteksi leksikon sanggahan (*Riedel et al.*, pemenang UCL FNC-1).
   - `ordered_bigram_ratio`: Keselarasan urutan kata bigram judul (*Yoon et al.*, AAAI 2019).
6. **Entity & Location Conflict**: Diskrepansi kanonikal wilayah, tokoh publik, dan lembaga pemerintah.
7. **Adaptive Discourse Inversion Scopes (4 Fitur Baru Penekan False Negatives)**:
   - `f_verb_neg`: Resolusi cakupan negasi verba (*local verb negation scope conflict*, e.g., larang vs tidak melarang).
   - `f_existential`: Konflik polaritas eksistensial (*ada* vs *tidak ada*).
   - `f_composite`: Diskrepansi pengikatan entitas komposit (*composite entity-location misbinding*, e.g., Bea Cukai Jember vs Bea Cukai Banyuwangi).
   - `f_directional`: Inversi arah peran gramatikal (*directional role inversion*, e.g., sumbang ke vs sumbang dari).

In [5]:
def extract_features(df_titles, df_contents, df_fs,
                     cos_wc, cos_wfs, cos_cc, cos_cfs,
                     bm25_c, bm25_fs,
                     cos_lsa_c, cos_lsa_fs,
                     title_tfidf_matrix):
    feats = []
    best_win_texts = []
    N_items = len(df_titles)
    
    for i in range(N_items):
        t_raw = df_titles[i]
        c_raw = df_contents[i]
        t_lower = t_raw.lower()
        c_lower = c_raw.lower()
        fs_lower = df_fs[i].lower()
        
        t_words = t_raw.split()
        c_words = c_raw.split()
        t_tokens = set(re.findall(r'\w+', t_lower))
        c_tokens = set(re.findall(r'\w+', c_lower))
        fs_tokens = set(re.findall(r'\w+', fs_lower))
        
        useful_t = [w for w in t_tokens if w not in STOP_WORDS and len(w) > 2]
        
        # 1. Canonical Locations
        t_locs = get_canonical_locations(t_lower)
        c_locs = get_canonical_locations(c_lower)
        fs_locs = get_canonical_locations(fs_lower)
        loc_missing = sum(1 for loc in t_locs if loc not in c_locs and not any(p in c_tokens for p in loc.split()))
        loc_mismatch = loc_missing / (len(t_locs) + 1e-9) if t_locs else 0.0
        loc_conflict = 1.0 if (t_locs and fs_locs and loc_missing > 0 and not t_locs.intersection(fs_locs)) else 0.0
        loc_c_conflict = 1.0 if (t_locs and c_locs and loc_missing > 0 and not t_locs.intersection(c_locs)) else 0.0
        
        # 2. Canonical Figures
        t_figs = get_canonical_figures(t_lower)
        c_figs = get_canonical_figures(c_lower)
        fig_missing = sum(1 for fig in t_figs if fig not in c_figs and not any(p in c_tokens for p in fig.split()))
        fig_mismatch = fig_missing / (len(t_figs) + 1e-9) if t_figs else 0.0
        fig_conflict = 1.0 if (t_figs and c_figs and fig_missing > 0 and not t_figs.intersection(c_figs)) else 0.0
        
        # 3. Institutions
        t_inst = {w for w in t_tokens if w in INSTITUTIONS}
        inst_missing = len([inst for inst in t_inst if inst not in c_tokens])
        inst_mismatch = inst_missing / (len(t_inst) + 1e-9) if t_inst else 0.0
        
        # 4. Token & Stemming Coverage
        exact_match = sum(1 for w in useful_t if w in c_tokens)
        c_stems = set(light_stem(w) for w in c_tokens if len(w) > 2)
        stem_match = sum(1 for w in useful_t if (w in c_tokens or light_stem(w) in c_stems))
        tok_cov_exact = exact_match / (len(useful_t) + 1e-9) if useful_t else 1.0
        tok_cov_stem = stem_match / (len(useful_t) + 1e-9) if useful_t else 1.0
        stem_gain = tok_cov_stem - tok_cov_exact
        
        # 5. Multi-Scale Discourse Windows (25, 50, 80 token)
        lead_25_tokens = set(re.findall(r'\w+', " ".join(c_words[:25]).lower()))
        lead_50_tokens = set(re.findall(r'\w+', " ".join(c_words[:50]).lower()))
        lead_80_tokens = set(re.findall(r'\w+', " ".join(c_words[:80]).lower()))
        
        lead_25_cov = len([w for w in useful_t if w in lead_25_tokens]) / (len(useful_t) + 1e-9) if useful_t else 1.0
        lead_50_cov = len([w for w in useful_t if w in lead_50_tokens]) / (len(useful_t) + 1e-9) if useful_t else 1.0
        lead_80_cov = len([w for w in useful_t if w in lead_80_tokens]) / (len(useful_t) + 1e-9) if useful_t else 1.0
        
        windows_50 = []
        w_size = 50
        step = 20
        for s in range(0, max(1, len(c_words)), step):
            chunk = " ".join(c_words[s:s+w_size])
            if len(chunk.split()) >= 5:
                windows_50.append(chunk)
        if not windows_50:
            windows_50 = [c_raw]
            
        best_overlap_50 = 0
        best_win_idx_50 = 0
        overlap_counts = []
        for w_idx, win in enumerate(windows_50):
            wtoks = set(re.findall(r'\w+', win.lower()))
            overlap = len([w for w in useful_t if w in wtoks])
            overlap_counts.append(overlap)
            if overlap > best_overlap_50:
                best_overlap_50 = overlap
                best_win_idx_50 = w_idx
                
        best_50_cov = best_overlap_50 / (len(useful_t) + 1e-9) if useful_t else 1.0
        best_win_pos_norm = best_win_idx_50 / (len(windows_50) + 1e-9)
        best_minus_lead = best_50_cov - lead_50_cov
        best_win_texts.append(windows_50[best_win_idx_50])
        
        sorted_overlaps = sorted(overlap_counts, reverse=True)
        top1_overlap = sorted_overlaps[0] if sorted_overlaps else 0
        top2_overlap = sorted_overlaps[1] if len(sorted_overlaps) > 1 else top1_overlap
        top1_top2_contrast = (top1_overlap - top2_overlap) / (len(useful_t) + 1e-9) if useful_t else 0.0
        
        # 6. Targeted Fact Guards
        head_ent = ""
        for j, w in enumerate(t_words[:4]):
            clean_w = re.sub(r'\W+', '', w)
            if clean_w and clean_w.lower() not in STOP_WORDS and len(clean_w) > 2:
                if w[0].isupper():
                    if j+1 < len(t_words) and t_words[j+1][0].isupper():
                        next_w = re.sub(r'\W+', '', t_words[j+1])
                        head_ent = f"{clean_w.lower()} {next_w.lower()}"
                    else:
                        head_ent = clean_w.lower()
                break
        head_entity_absence = 1.0 if (head_ent and head_ent not in c_lower) else 0.0
        
        num_units = re_num_unit.findall(t_lower)
        if num_units:
            matches = 0
            for num, unit in num_units:
                pat = rf"{re.escape(num)}\s+(?:\w+\s+){{0,2}}{re.escape(unit)}"
                if re.search(pat, c_lower):
                    matches += 1
            num_unit_match_ratio = matches / len(num_units)
        else:
            num_unit_match_ratio = 1.0
            
        t_negs = t_tokens & NEGATION_WORDS
        best_w_tokens = set(re.findall(r'\w+', windows_50[best_win_idx_50].lower()))
        win_negs = best_w_tokens & NEGATION_WORDS
        lead_negs = lead_50_tokens & NEGATION_WORDS
        neg_inversion = 1.0 if (bool(t_negs) != bool(win_negs)) else 0.0
        lead_neg_asymmetry = 1.0 if (bool(t_negs) != bool(lead_negs)) else 0.0
        
        refuting_matches = len(re_refuting.findall(windows_50[best_win_idx_50]))
        refuting_score = float(min(refuting_matches, 3))
        
        t_clean_words = [w for w in re.findall(r'\b[a-z0-9]+\b', t_lower) if len(w) > 2 and w not in STOP_WORDS]
        if len(t_clean_words) >= 2:
            title_ordered_bgs = [f"{t_clean_words[j]} {t_clean_words[j+1]}" for j in range(len(t_clean_words)-1)]
            matched_bgs = sum(1 for bg in title_ordered_bgs if bg in c_lower)
            ordered_bigram_ratio = matched_bgs / len(title_ordered_bgs)
        else:
            ordered_bigram_ratio = 1.0
            
        # 7. Antonym & Conflict
        ada_conflict = 0.0
        if ('ada' in t_tokens and not t_negs):
            if 'tidak ada' in fs_lower or 'tak ada' in fs_lower:
                ada_conflict = 1.0
        elif ('tidak ada' in t_lower or 'tak ada' in t_lower):
            if 'ada' in fs_tokens and 'tidak ada' not in fs_lower and 'tak ada' not in fs_lower:
                ada_conflict = 1.0
                
        antonym_conflict = 0.0
        for w1, w2 in EXPANDED_ANTONYM_PAIRS:
            if (w1 in t_tokens and w2 in c_tokens) or (w2 in t_tokens and w1 in c_tokens):
                antonym_conflict = 1.0
                break
                
        # 8. Numbers Comprehensive
        t_subst_nums = extract_numbers_comprehensive(t_raw)
        c_subst_nums = extract_numbers_comprehensive(c_raw)
        num_missing_subst = len(t_subst_nums - c_subst_nums)
        num_cov_subst = len(t_subst_nums.intersection(c_subst_nums)) / (len(t_subst_nums) + 1e-9) if t_subst_nums else 1.0
        
        # 9. Capital Words & Lengths
        is_title_case = sum(1 for w in t_words if w and w[0].isupper()) > 0.8 * len(t_words)
        if not is_title_case:
            cap_words = [w for w in t_words if len(w) > 1 and w[0].isupper() and w.lower() not in STOP_WORDS]
            cap_missing = sum(1 for w in cap_words if w.lower() not in c_tokens)
            cap_missing_fs = sum(1 for w in cap_words if w.lower() not in fs_tokens)
            cap_cov_c = 1.0 - (cap_missing / (len(cap_words) + 1e-9)) if cap_words else 1.0
            cap_cov_fs = 1.0 - (cap_missing_fs / (len(cap_words) + 1e-9)) if cap_words else 1.0
        else:
            cap_cov_c = 1.0
            cap_cov_fs = 1.0
            
        head_content_words = [w.lower() for w in t_words if w.lower() not in STOP_WORDS][:3]
        head_missing_c = sum(1 for hw in head_content_words if hw not in c_tokens)
        head_cov_c = 1.0 - (head_missing_c / (len(head_content_words) + 1e-9)) if head_content_words else 1.0
        
        len_ratio = len(c_raw) / (len(t_raw) + 1e-9)
        
        # 10. Adaptive Discourse Inversion Scopes (4 Fitur Baru Penekan False Negatives)
        t_lower_words = t_lower.split()
        lead_words = c_lower.split()[:60]
        lead_text = " ".join(lead_words)
        
        # 10a. Verb Negation Scope Conflict
        v_conf = 0.0
        for idx, w in enumerate(t_lower_words):
            is_action = (w in INHERENT_NEG_VERBS) or (len(w) > 4 and w.startswith(('me', 'di', 'ber', 'ter'))) or w in ['bisa', 'dapat', 'sebabkan', 'izinkan', 'bolehkan']
            if is_action:
                t_is_neg = (w in INHERENT_NEG_VERBS) or (idx > 0 and t_lower_words[idx-1] in NEG_WORDS) or (idx > 1 and t_lower_words[idx-2] in NEG_WORDS)
                c_instances = []
                for j, cw in enumerate(lead_words):
                    if cw == w or (len(w) >= 5 and len(cw) >= 5 and w[:5] == cw[:5]):
                        c_neg = (cw in INHERENT_NEG_VERBS) or (j > 0 and lead_words[j-1] in NEG_WORDS) or (j > 1 and lead_words[j-2] in NEG_WORDS)
                        c_instances.append(c_neg)
                if c_instances:
                    if not t_is_neg and all(c_instances):
                        v_conf = 1.0
                        break
                    elif t_is_neg and not any(c_instances):
                        v_conf = 1.0
                        break
                        
        # 10b. Existential Polarity Conflict
        e_conf = 0.0
        t_has_ada = 'ada ' in t_lower[:25] or t_lower.startswith('ada ')
        t_neg_ada = 'tidak ada' in t_lower[:30] or 'tak ada' in t_lower[:30]
        c_has_ada = 'ada ' in lead_text[:30] or lead_text.startswith('ada ')
        c_neg_ada = 'tidak ada' in lead_text[:35] or 'tak ada' in lead_text[:35]
        if (t_has_ada and not t_neg_ada) and c_neg_ada:
            e_conf = 1.0
        elif t_neg_ada and (c_has_ada and not c_neg_ada):
            e_conf = 1.0
            
        # 10c. Composite Entity Conflict
        comp_conf = 0.0
        m_inst = list(re_inst_marker.finditer(t_lower))
        m_loc_t = list(LOC_REGEX.finditer(t_lower))
        if m_inst and m_loc_t:
            for inst in m_inst:
                for loc in m_loc_t:
                    if 0 <= loc.start() - inst.end() <= 30 or 0 <= inst.start() - loc.end() <= 30:
                        inst_name = inst.group(0)
                        loc_name_t = loc.group(0)
                        idx_c = c_lower.find(inst_name)
                        if idx_c != -1:
                            surrounding = c_lower[max(0, idx_c-30):min(len(c_lower), idx_c+len(inst_name)+30)]
                            locs_c = [m.group(0) for m in LOC_REGEX.finditer(surrounding)]
                            if locs_c and loc_name_t not in locs_c:
                                comp_conf = 1.0
                                break
                if comp_conf == 1.0:
                    break
                    
        # 10d. Directional Role Inversion
        dir_conf = 0.0
        prep_m = list(re.finditer(r'\b(ke|kepada)\s+([a-zA-Z]+)', t_lower))
        if prep_m:
            for m in prep_m:
                target = m.group(2)
                if len(target) > 3 and target not in ['mana', 'sana', 'depan', 'dalam', 'luar']:
                    actor_pat = rf"\b{re.escape(target)}\s+(?:akan|telah|sudah|kembali|bakal|siap|resmi|mulai|menyumbang|menyerahkan|memberi)"
                    if re.search(actor_pat, lead_text):
                        dir_conf = 1.0
                        break
        
        feats.append([
            cos_wc[i], cos_wfs[i], cos_cc[i], cos_cfs[i],
            bm25_c[i], bm25_fs[i],
            cos_lsa_c[i], cos_lsa_fs[i],
            best_50_cov, best_win_pos_norm, best_minus_lead,
            lead_25_cov, lead_50_cov, lead_80_cov, top1_top2_contrast,
            loc_mismatch, loc_conflict, loc_c_conflict,
            fig_mismatch, fig_conflict,
            inst_mismatch,
            tok_cov_exact, tok_cov_stem, stem_gain, head_cov_c,
            head_entity_absence, num_unit_match_ratio,
            neg_inversion, lead_neg_asymmetry, ada_conflict, antonym_conflict,
            refuting_score, ordered_bigram_ratio,
            num_missing_subst, num_cov_subst,
            cap_cov_c, cap_cov_fs, len_ratio, len(t_words), len(c_words),
            v_conf, e_conf, comp_conf, dir_conf
        ])
    
    # Ultra-fast Batch TF-IDF transform for best focal window cosine
    win_matrix = tfidf_word.transform(best_win_texts)
    best_win_tfidf_cos = np.asarray((title_tfidf_matrix.multiply(win_matrix)).sum(axis=1)).flatten()
    
    feature_matrix = np.array(feats)
    # Insert best_win_tfidf_cos at column index 8
    feature_matrix = np.insert(feature_matrix, 8, best_win_tfidf_cos, axis=1)
    return feature_matrix

print("Mengekstrak 45 fitur teroptimasi (Opsi A) untuk data train dan test...")
X_train = extract_features(
    train_titles, train_contents, train_fs,
    cos_w_c_tr, cos_w_fs_tr, cos_c_c_tr, cos_c_fs_tr,
    bm25_c_tr, bm25_fs_tr,
    cos_lsa_c_tr, cos_lsa_fs_tr,
    tr_t_w
)
X_test = extract_features(
    test_titles, test_contents, test_fs,
    cos_w_c_te, cos_w_fs_te, cos_c_c_te, cos_c_fs_te,
    bm25_c_te, bm25_fs_te,
    cos_lsa_c_te, cos_lsa_fs_te,
    te_t_w
)
y_train = train['label'].values

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"X_train shape: {X_train.shape} | X_test shape: {X_test.shape}")

Mengekstrak 41 fitur teroptimasi (Opsi A) untuk data train dan test...
X_train shape: (14397, 41) | X_test shape: (3603, 41)

## 5. Stratified 5-Fold Cross Validation & 4-Model Stacking Ensemble (45 Fitur)
Pada tahap ini, kita melatih ensemble 4 model yang saling melengkapi di atas 45 fitur terkalibrasi:
1. **XGBoost Classifier**: Pohon gradien boosting teratur dengan subsampling.
2. **HistGradientBoosting Classifier**: Pengambilan keputusan berbasis histogram leksikal.
3. **ExtraTrees Classifier**: Pohon acak berdensitas tinggi untuk menekan variansi.
4. **Logistic Regression**: Model linier kalibrasi pada ruang fitur terstandarisasi.

Meta-Learner menggabungkan out-of-fold probability predictions dari ke-4 model. Kemudian dilakukan pencarian threshold optimal $T^*$ dan evaluasi performa kuantil berbasis prior sebaran kelas kompetisi (~10% kelas 0).

In [6]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
sample_weights = np.ones(len(train))

xgb_oof = np.zeros(len(train))
hgb_oof = np.zeros(len(train))
et_oof = np.zeros(len(train))
lr_oof = np.zeros(len(train))

xgb_test = np.zeros(len(test))
hgb_test = np.zeros(len(test))
et_test = np.zeros(len(test))
lr_test = np.zeros(len(test))

for fold, (trn_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
    print(f"--- Training Fold {fold+1} (XGBoost, HistGradientBoosting, ExtraTrees, LogisticRegression) ---")
    
    clf_xgb = xgb.XGBClassifier(
        n_estimators=400, max_depth=5, learning_rate=0.035,
        subsample=0.8, colsample_bytree=0.8, random_state=SEED, eval_metric='logloss'
    )
    clf_xgb.fit(X_train[trn_idx], y_train[trn_idx], sample_weight=sample_weights[trn_idx])
    xgb_oof[val_idx] = clf_xgb.predict_proba(X_train[val_idx])[:, 1]
    xgb_test += clf_xgb.predict_proba(X_test)[:, 1] / 5.0
    
    clf_hgb = HistGradientBoostingClassifier(
        max_iter=300, learning_rate=0.035, max_depth=6, random_state=SEED
    )
    clf_hgb.fit(X_train[trn_idx], y_train[trn_idx], sample_weight=sample_weights[trn_idx])
    hgb_oof[val_idx] = clf_hgb.predict_proba(X_train[val_idx])[:, 1]
    hgb_test += clf_hgb.predict_proba(X_test)[:, 1] / 5.0
    
    clf_et = ExtraTreesClassifier(n_estimators=250, max_depth=12, random_state=SEED, n_jobs=-1)
    clf_et.fit(X_train[trn_idx], y_train[trn_idx], sample_weight=sample_weights[trn_idx])
    et_oof[val_idx] = clf_et.predict_proba(X_train[val_idx])[:, 1]
    et_test += clf_et.predict_proba(X_test)[:, 1] / 5.0
    
    clf_lr = LogisticRegression(C=0.5, max_iter=1000, random_state=SEED)
    clf_lr.fit(X_train_scaled[trn_idx], y_train[trn_idx], sample_weight=sample_weights[trn_idx])
    lr_oof[val_idx] = clf_lr.predict_proba(X_train_scaled[val_idx])[:, 1]
    lr_test += clf_lr.predict_proba(X_test_scaled)[:, 1] / 5.0

# Stacking Meta-Learner Cross Validation
meta_X = np.column_stack([xgb_oof, hgb_oof, et_oof, lr_oof])
meta_X_test = np.column_stack([xgb_test, hgb_test, et_test, lr_test])

meta_oof = np.zeros(len(train))
meta_test = np.zeros(len(test))

for trn, val in skf.split(meta_X, y_train):
    meta_clf = LogisticRegression(C=1.0, random_state=SEED)
    meta_clf.fit(meta_X[trn], y_train[trn])
    meta_oof[val] = meta_clf.predict_proba(meta_X[val])[:, 1]
    meta_test += meta_clf.predict_proba(meta_X_test)[:, 1] / 5.0

# Evaluasi 1: Pencarian Threshold Probabilitas Optimal
best_th, best_f1 = 0.810, 0.0
for th in np.arange(0.1, 0.95, 0.005):
    preds = (meta_oof >= th).astype(int)
    sc = f1_score(y_train, preds, average='macro')
    if sc > best_f1:
        best_f1, best_th = sc, th

print("\n" + "="*50)
print(f"HASIL KALIBRASI OPTIMAL THRESHOLD: {best_th:.3f} | MACRO F1: {best_f1:.4f}")
print("="*50)
oof_preds = (meta_oof >= best_th).astype(int)
print(classification_report(y_train, oof_preds, digits=4))
print("Confusion Matrix (Optimal Threshold):")
print(confusion_matrix(y_train, oof_preds))

# Evaluasi 2: Kalibrasi Quantile Prior (~10% Kelas 0)
prior_pct = 9.98
q_th = np.percentile(meta_oof, prior_pct)
oof_preds_q = (meta_oof >= q_th).astype(int)
q_f1 = f1_score(y_train, oof_preds_q, average='macro')
print("\n" + "="*50)
print(f"HASIL KALIBRASI QUANTILE PRIOR (Top {prior_pct}%): Cutoff Prob={q_th:.4f} | MACRO F1: {q_f1:.4f}")
print("="*50)
print(classification_report(y_train, oof_preds_q, digits=4))

--- Training Fold 1 (XGBoost, HistGradientBoosting, ExtraTrees, LogisticRegression) ---
--- Training Fold 2 (XGBoost, HistGradientBoosting, ExtraTrees, LogisticRegression) ---
--- Training Fold 3 (XGBoost, HistGradientBoosting, ExtraTrees, LogisticRegression) ---
--- Training Fold 4 (XGBoost, HistGradientBoosting, ExtraTrees, LogisticRegression) ---
--- Training Fold 5 (XGBoost, HistGradientBoosting, ExtraTrees, LogisticRegression) ---

HASIL KALIBRASI OPTIMAL THRESHOLD: 0.825 | MACRO F1: 0.7089
              precision    recall  f1-score   support

           0     0.5458    0.4106    0.4686      1437
           1     0.9364    0.9621    0.9491     12960

    accuracy                         0.9071     14397
   macro avg     0.7411    0.6863    0.7089     14397
weighted avg     0.8974    0.9071    0.9011     14397

Confusion Matrix (Optimal Threshold):
[[  590   847]
 [  491 12469]]

HASIL KALIBRASI QUANTILE PRIOR (Top 9.98%): Cutoff Prob=0.8650 | MACRO F1: 0.6958
              precis

## 6. Prediksi Data Uji (`test.csv`) & Pembuatan Submission
Menerapkan threshold terkalibrasi ke probabilitas prediksi data uji dan mengekspor hasilnya ke `submission_baseline.csv`.

In [7]:
test_preds = (meta_test >= best_th).astype(int)

submission = pd.DataFrame({'id': test['id'], 'label': test_preds})
print("--- Distribusi Prediksi Submission (Optimal Threshold) ---")
print(submission['label'].value_counts())
print(f"Persentase Kelas 1: {(submission['label']==1).mean()*100:.2f}%")
print(f"Persentase Kelas 0: {(submission['label']==0).mean()*100:.2f}%")

submission.to_csv('submission_baseline.csv', index=False)
print("\nSubmission file saved successfully: submission_baseline.csv!")
print(f"Total rows: {len(submission)} | Null values: {submission.isnull().sum().sum()}")

--- Distribusi Prediksi Submission (Optimal Threshold) ---
label
1    3342
0     261
Name: count, dtype: int64
Persentase Kelas 1: 92.76%
Persentase Kelas 0: 7.24%

Submission file saved successfully: submission_baseline.csv!
Total rows: 3603 | Null values: 0